<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Final_XLSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Dynamic Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42)
np.random.seed(42)

def init_weights(m):
    """تطبيق Xavier Initialization وتصفير الـ bias لطبقات Linear"""
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0.0)

class NativesLSTMLayer(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(NativesLSTMLayer, self).__init__()
        self.hidden_dim = hidden_dim
        self.W_x = nn.Linear(input_dim, 4 * hidden_dim, bias=True)
        self.W_h = nn.Linear(hidden_dim, 4 * hidden_dim, bias=False)

        # تطبيق التهيأة على الطبقات الداخلية
        init_weights(self.W_x)
        init_weights(self.W_h)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        c = torch.zeros(batch_size, self.hidden_dim, device=x.device)
        outputs = []
        for t in range(seq_len):
            x_t = x[:, t, : ]
            gates = self.W_x(x_t) + self.W_h(h)
            i_gate, f_gate, c_gate, o_gate = gates.chunk(4, dim=1)

            i_t = torch.exp(torch.clamp(i_gate, -5.0, 5.0))
            f_t = torch.exp(torch.clamp(f_gate, -5.0, 5.0))

            c_tilde = torch.tanh(c_gate)
            c = f_t * c + i_t * c_tilde
            o_t = torch.sigmoid(o_gate)
            h = o_t * torch.tanh(c)
            outputs.append(h.unsqueeze(1))
        return torch.cat(outputs, dim=1)

class JointxLSTMAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, dropout_prob=0.2):
        super(JointxLSTMAutoencoder, self).__init__()

        # Encoder Stack
        self.encoder_xlstm1 = NativesLSTMLayer(input_dim, 32)
        self.encoder_xlstm2 = NativesLSTMLayer(32, 16)
        self.bottleneck = nn.Linear(16, latent_dim)
        self.encoder_dropout = nn.Dropout(dropout_prob)

        # Decoder Stack (Reconstruction Head)
        self.decoder_xlstm1 = NativesLSTMLayer(latent_dim, 16)
        self.decoder_xlstm2 = NativesLSTMLayer(16, 32)
        self.reconstruct = nn.Linear(32, input_dim)
        self.decoder_dropout = nn.Dropout(dropout_prob)

        # Prediction Head (Supervised branch targeting Carbon Emission)
        self.predictor_head = nn.Linear(latent_dim, 1)

        # تطبيق Xavier Initialization على الطبقات الخطيّة المستقلة
        init_weights(self.bottleneck)
        init_weights(self.reconstruct)
        init_weights(self.predictor_head)

    def forward(self, x):
        # Pass through Encoder
        encoded = self.encoder_xlstm1(x)
        encoded = self.encoder_dropout(encoded)
        encoded = self.encoder_xlstm2(encoded)
        latent = self.bottleneck(encoded[:, -1, :])

        # Branch 1: Reconstruct Features X
        decoded_input = latent.unsqueeze(1).repeat(1, x.size(1), 1)
        decoded = self.decoder_xlstm1(decoded_input)
        decoded = self.decoder_dropout(decoded)
        decoded = self.decoder_xlstm2(decoded)
        reconstructed_output = torch.sigmoid(self.reconstruct(decoded))

        # Branch 2: Predict Carbon Target Y directly from latent features
        predicted_carbon = self.predictor_head(latent)

        return reconstructed_output, predicted_carbon, latent

#Cell 2: (Data Preprocessing & Splitting)

In [2]:
FILE_PATH = '/content/rural_carbon_dataset.csv'
df = pd.read_csv(FILE_PATH)
df_processed = df.copy()

# Feature Encoding & Circular Months Transformations
crop_encoder = LabelEncoder()
df_processed['Crop_Type_Encoded'] = crop_encoder.fit_transform(df_processed['Crop_Type'])
df_processed['Month_sin'] = np.sin(2 * np.pi * df_processed['Month'] / 12)
df_processed['Month_cos'] = np.cos(2 * np.pi * df_processed['Month'] / 12)
df_processed['Livestock_Total'] = df_processed['Livestock_Cows'] + df_processed['Livestock_Pigs']
df_processed['Energy_per_Area'] = df_processed['Household_Energy_kWh'] / (df_processed['Crop_Area_ha'] + 1)
df_processed['Fertilizer_per_Area'] = df_processed['Fertilizer_Usage_kg'] / (df_processed['Crop_Area_ha'] + 1)

feature_cols = [
    'Month_sin', 'Month_cos', 'Crop_Type_Encoded', 'Crop_Area_ha', 'Livestock_Total',
    'Household_Energy_kWh', 'Renewable_Energy_Fraction', 'Temperature_C',
    'Rainfall_mm', 'Energy_per_Area', 'Fertilizer_per_Area'
]

X = df_processed[feature_cols].values.astype(np.float32)
y = df_processed['Carbon_Emission_tCO2'].values.astype(np.float32).reshape(-1, 1)


X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Unified MinMax Normalization for Inputs X & Target Y
scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_val_scaled = scaler_y.transform(y_val)

# Reshaping for Sequential 3D Layout
X_train_3d = np.expand_dims(X_train_scaled, axis=1)
X_val_3d = np.expand_dims(X_val_scaled, axis=1)

# Package both X and Y into the Loader for Joint Training
train_dataset = TensorDataset(torch.tensor(X_train_3d), torch.tensor(y_train_scaled))
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_inputs_X = torch.tensor(X_val_3d).to(device)
val_targets_y = torch.tensor(y_val_scaled).to(device)

print(f"Data split into Train and Validation sets successfully. Active device: {device}")

Data split into Train and Validation sets successfully. Active device: cpu


#Cell 3:(Training Configurations & Optimizers)

In [4]:
feat_dim = X_train_3d.shape[2]
encoding_dim = 8  # Compressed Bottleneck Dimension

model = JointxLSTMAutoencoder(input_dim=feat_dim, latent_dim=encoding_dim, dropout_prob=0.2).to(device)
criterion_recon = nn.MSELoss()  # For X reconstruction
criterion_pred = nn.MSELoss()   # For Y prediction

# استخدام Pure Adam Optimizer بدون مدخلات خارجية
optimizer = optim.Adam(model.parameters(), lr=0.001)

# إضافة Learning Rate Scheduler يعتمد على الـ validation loss
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

epochs = 100
best_val_loss = float('inf')
patience, patience_counter = 20, 0
alpha = 1.0  # Loss weighting parameter
checkpoint_path = 'best_joint_xlstm_ae_checkpoint.pth'

print("Hyperparameters and Optimization components successfully configured.")

Hyperparameters and Optimization components successfully configured.


#Cell 4: (Structured Training & Validation Loop)

In [5]:
print("Commencing Joint Unsupervised-Supervised xLSTM Autoencoder Training...")
print("-" * 85)

for epoch in range(epochs):
    # ==================== STAGE 1: TRAINING ====================
    model.train()
    train_total_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        recon_out, pred_carbon, _ = model(batch_x)

        # Calculate joint Multi-task Losses
        loss_recon = criterion_recon(recon_out, batch_x)
        loss_pred = criterion_pred(pred_carbon, batch_y)
        loss_total = loss_recon + (alpha * loss_pred)

        loss_total.backward()

        # إضافة الـ Gradient Clipping بقيمة 5.0 لمنع تضخم المشتقات
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

        optimizer.step()

        train_total_loss += loss_total.item() * batch_x.size(0)
    train_total_loss /= len(train_loader.dataset)

    # ==================== STAGE 2: VALIDATION ====================
    model.eval()
    val_recon_loss = 0.0
    val_pred_loss = 0.0
    with torch.no_grad():
        val_recon, val_pred, _ = model(val_inputs_X)
        val_recon_loss = criterion_recon(val_recon, val_inputs_X).item()
        val_pred_loss = criterion_pred(val_pred, val_targets_y).item()
        val_total_loss = val_recon_loss + (alpha * val_pred_loss)

    # تحديث الـ Scheduler بناءً على الـ Validation Loss
    scheduler.step(val_total_loss)

    # ==================== STAGE 3: CHECKPOINT & EARLY STOPPING ====================
    if val_total_loss < best_val_loss:
        best_val_loss = val_total_loss
        patience_counter = 0
        # حفظ أفضل نموذج بناءً على الـ Validation Loss مع checkpoint واضح
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, checkpoint_path)
    else:
        patience_counter += 1

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1:03d}/{epochs}] -> Train Loss: {train_total_loss:.6f} | Val Recon Loss: {val_recon_loss:.6f} | Val Pred Loss: {val_pred_loss:.6f} | Total Val Loss: {val_total_loss:.6f}")

    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch+1} based on Validation Loss to prevent Overfitting.")
        break

print("-" * 85)
print("Training finalized.")

Commencing Joint Unsupervised-Supervised xLSTM Autoencoder Training...
-------------------------------------------------------------------------------------
Epoch [001/100] -> Train Loss: 0.168321 | Val Recon Loss: 0.095675 | Val Pred Loss: 0.026709 | Total Val Loss: 0.122384
Epoch [010/100] -> Train Loss: 0.076574 | Val Recon Loss: 0.058751 | Val Pred Loss: 0.011184 | Total Val Loss: 0.069934
Epoch [020/100] -> Train Loss: 0.059651 | Val Recon Loss: 0.040763 | Val Pred Loss: 0.011174 | Total Val Loss: 0.051937
Epoch [030/100] -> Train Loss: 0.050015 | Val Recon Loss: 0.030321 | Val Pred Loss: 0.011138 | Total Val Loss: 0.041459
Epoch [040/100] -> Train Loss: 0.041517 | Val Recon Loss: 0.022139 | Val Pred Loss: 0.010992 | Total Val Loss: 0.033132
Epoch [050/100] -> Train Loss: 0.038449 | Val Recon Loss: 0.018871 | Val Pred Loss: 0.011084 | Total Val Loss: 0.029955
Epoch [060/100] -> Train Loss: 0.037046 | Val Recon Loss: 0.017535 | Val Pred Loss: 0.011135 | Total Val Loss: 0.028670
Epo

#Cell 5:(Comprehensive Evaluation & Metrics)

In [6]:
def calculate_mape(y_true, y_pred):
    """حساب مقياس Mean Absolute Percentage Error بشكل آمن"""
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model weights from epoch {checkpoint['epoch']}")

model.eval()
with torch.no_grad():
    final_recon, final_pred, final_latent = model(val_inputs_X)

y_true_np = val_targets_y.cpu().numpy().flatten()
y_pred_np = final_pred.cpu().numpy().flatten()

X_true_flat = X_val_3d.reshape(-1, feat_dim)
X_recon_flat = final_recon.cpu().numpy().reshape(-1, feat_dim)

# 1.(Reconstruction Metrics)
recon_r2 = r2_score(X_true_flat, X_recon_flat)

# 2.(Prediction Metrics)
pred_r2 = r2_score(y_true_np, y_pred_np)
pred_rmse = np.sqrt(mean_squared_error(y_true_np, y_pred_np))
pred_mae = mean_absolute_error(y_true_np, y_pred_np)
pred_mape = calculate_mape(y_true_np, y_pred_np)

print("\n========== Official Joint xLSTM Architecture Evaluation (Validation Set) ==========")
print(f"Reconstruction Task Test R² (X Integrity)     = {recon_r2:.6f}")
print(f"Normalized Prediction Task Test R² (Carbon Y) = {pred_r2:.6f}")
print(f"Normalized Prediction Task Test RMSE          = {pred_rmse:.6f}")
print(f"Normalized Prediction Task Test MAE           = {pred_mae:.6f}")
print(f"Normalized Prediction Task Test MAPE          = {pred_mape:.2f}%")
print(f"Conditioned Latent Space Shape                = {final_latent.shape}")
print("===================================================================================")

Loaded best model weights from epoch 100

========== Official Joint xLSTM Architecture Evaluation (Validation Set) ==========
Reconstruction Task Test R² (X Integrity)     = 0.865962
Normalized Prediction Task Test R² (Carbon Y) = 0.470992
Normalized Prediction Task Test RMSE          = 0.105389
Normalized Prediction Task Test MAE           = 0.082997
Normalized Prediction Task Test MAPE          = 16.85%
Conditioned Latent Space Shape                = torch.Size([600, 8])
